# Artifact 24 — discriminant-surface picture candidates

This notebook is a lightweight companion to the earlier Binder notebook. It
contains several rotatable Plotly pictures of the range discriminant /
nonproperness surface and its omitted curve.

It is intentionally **self-contained**: only NumPy and Plotly are required.
The coordinates are the original raw range coordinates
\[
F=(P,Q,R),
\]
not the normalized ordering \(\widehat F=(R/2,Q,P)\).

The discriminant surface is parametrized by
\[
(P,Q,R)
=
\left(
\rho^2-c\rho^3,\;
4\rho-3c\rho^2,\;
c
\right),
\]
and the missing curve is
\[
(P,Q,R)
=
\left(
\frac{\rho^2}{3},\;
2\rho,\;
\frac{2}{3\rho}
\right).
\]

Run the notebook from top to bottom, rotate each picture, and note the
candidate number you prefer.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"

P_RANGE = (-0.8, 5.2)
Q_RANGE = (-7.0, 7.0)
R_RANGE = (-1.5, 5.0)

PLOT_CONFIG = {
    "scrollZoom": True,
    "displaylogo": False,
    "responsive": True,
}

print("NumPy:", np.__version__)
print("Plotly renderer:", pio.renderers.default)

## Shared plotting and verification helpers

In [ ]:
def discriminant(P, Q, R):
    # Ulam-paper discriminant in raw (P,Q,R) coordinates.
    return Q**2 - 16.0*P - Q**3*R + 18.0*P*Q*R - 27.0*P**2*R**2


def surface_coordinates(rho, c):
    # Parametrization of the discriminant/nonproperness surface.
    P = rho**2 - c*rho**3
    Q = 4.0*rho - 3.0*c*rho**2
    R = c
    return P, Q, R


def missing_coordinates(rho):
    # Parametrization of the omitted triple-root curve.
    P = rho**2 / 3.0
    Q = 2.0*rho
    R = 2.0 / (3.0*rho)
    return P, Q, R


def clip_xyz(P, Q, R):
    # Replace points outside the common comparison window by NaN.
    P, Q, R = np.broadcast_arrays(
        np.asarray(P, dtype=float),
        np.asarray(Q, dtype=float),
        np.asarray(R, dtype=float),
    )
    visible = (
        np.isfinite(P) & np.isfinite(Q) & np.isfinite(R)
        & (P >= P_RANGE[0]) & (P <= P_RANGE[1])
        & (Q >= Q_RANGE[0]) & (Q <= Q_RANGE[1])
        & (R >= R_RANGE[0]) & (R <= R_RANGE[1])
    )
    return (
        np.where(visible, P, np.nan),
        np.where(visible, Q, np.nan),
        np.where(visible, R, np.nan),
    )


def common_layout(title):
    return dict(
        title=title,
        scene=dict(
            xaxis=dict(title="P", range=list(P_RANGE), autorange=False),
            yaxis=dict(title="Q", range=list(Q_RANGE), autorange=False),
            zaxis=dict(title="R", range=list(R_RANGE), autorange=False),
            aspectmode="data",
            camera=dict(eye=dict(x=1.55, y=-1.80, z=0.85)),
        ),
        margin=dict(l=0, r=0, b=0, t=48),
        legend=dict(itemsizing="constant"),
    )


def add_surface_patch(fig, opacity=0.14, grid_size=150):
    rho_values = np.linspace(-3.4, 3.4, grid_size)
    c_values = np.linspace(R_RANGE[0], R_RANGE[1], grid_size)
    rho, c = np.meshgrid(rho_values, c_values)

    P, Q, R = surface_coordinates(rho, c)
    P, Q, R = clip_xyz(P, Q, R)

    fig.add_trace(
        go.Surface(
            x=P,
            y=Q,
            z=R,
            surfacecolor=rho,
            colorscale="Cividis",
            opacity=opacity,
            showscale=False,
            name="discriminant surface",
            hovertemplate=(
                "<b>discriminant surface</b>"
                "<br>P=%{x:.5g}"
                "<br>Q=%{y:.5g}"
                "<br>R=%{z:.5g}"
                "<extra></extra>"
            ),
        )
    )


def add_missing_curve(fig, width=11):
    rho_abs = np.geomspace(1.0e-4, 1.0e4, 24000)

    for sign, label in (
        (+1.0, "missing curve, rho > 0"),
        (-1.0, "missing curve, rho < 0"),
    ):
        rho = sign * rho_abs
        P, Q, R = missing_coordinates(rho)
        P, Q, R = clip_xyz(P, Q, R)

        fig.add_trace(
            go.Scatter3d(
                x=P,
                y=Q,
                z=R,
                mode="lines",
                line=dict(color="#ff00aa", width=width),
                name=label,
                legendgroup="missing-curve",
                customdata=rho,
                hovertemplate=(
                    "<b>missing curve</b>"
                    "<br>rho=%{customdata:.6g}"
                    "<br>P=%{x:.6g}"
                    "<br>Q=%{y:.6g}"
                    "<br>R=%{z:.6g}"
                    "<extra></extra>"
                ),
            )
        )


def show_figure(fig, title):
    fig.update_layout(**common_layout(title))
    fig.show(config=PLOT_CONFIG)

In [ ]:
# Numerical smoke checks for every formula used below.

rng = np.random.default_rng(20260729)
rho_test = rng.uniform(-3.0, 3.0, 2000)
c_test = rng.uniform(-1.5, 5.0, 2000)

P_test, Q_test, R_test = surface_coordinates(rho_test, c_test)
surface_residual = np.max(np.abs(discriminant(P_test, Q_test, R_test)))

rho_missing = np.concatenate([
    -np.geomspace(0.05, 20.0, 1000),
     np.geomspace(0.05, 20.0, 1000),
])
Pm, Qm, Rm = missing_coordinates(rho_missing)
missing_residual = np.max(np.abs(discriminant(Pm, Qm, Rm)))

P0 = rng.uniform(-0.5, 4.5, 1000)
rho_nonzero = rng.choice([-1.0, 1.0], 1000) * rng.uniform(0.1, 5.0, 1000)
c_from_P = 1.0/rho_nonzero - P0/rho_nonzero**3
Q_from_P = rho_nonzero + 3.0*P0/rho_nonzero
P_back, Q_back, R_back = surface_coordinates(rho_nonzero, c_from_P)

Q0 = rng.uniform(-6.0, 6.0, 1000)
c_from_Q = (4.0*rho_nonzero - Q0) / (3.0*rho_nonzero**2)
P_from_Q = rho_nonzero*(Q0 - rho_nonzero)/3.0
P_back2, Q_back2, R_back2 = surface_coordinates(rho_nonzero, c_from_Q)

fixed_P_error = max(
    np.max(np.abs(P_back - P0)),
    np.max(np.abs(Q_back - Q_from_P)),
)
fixed_Q_error = max(
    np.max(np.abs(P_back2 - P_from_Q)),
    np.max(np.abs(Q_back2 - Q0)),
)

print(f"surface discriminant residual: {surface_residual:.3e}")
print(f"missing-curve residual:        {missing_residual:.3e}")
print(f"fixed-P rearrangement error:   {fixed_P_error:.3e}")
print(f"fixed-Q rearrangement error:   {fixed_Q_error:.3e}")

assert surface_residual < 1.0e-8
assert missing_residual < 1.0e-8
assert fixed_P_error < 1.0e-10
assert fixed_Q_error < 1.0e-10

print("Formula checks passed.")

## Candidate 1 — translucent surface with missing curve

This is the simplest global picture. It is useful as a reference, but the
surface patch may obscure its internal geometry.

In [ ]:
fig1 = go.Figure()
add_surface_patch(fig1, opacity=0.26)
add_missing_curve(fig1, width=12)
show_figure(fig1, "Candidate 1: surface patch and missing curve")

## Candidate 2 — straight ruling lines, with a faint surface

For fixed \(\rho\), varying \(c=R\) gives a straight line on the surface.
This is probably the clearest structural picture.

In [ ]:
fig2 = go.Figure()
add_surface_patch(fig2, opacity=0.10)

rho_values = [-3.0, -2.5, -2.0, -1.5, -1.0, -0.55,
               0.55, 1.0, 1.5, 2.0, 2.5, 3.0]
c = np.linspace(R_RANGE[0], R_RANGE[1], 1000)

for rho0 in rho_values:
    P, Q, R = surface_coordinates(rho0, c)
    P, Q, R = clip_xyz(P, Q, R)
    fig2.add_trace(
        go.Scatter3d(
            x=P, y=Q, z=R,
            mode="lines",
            line=dict(width=5),
            name=f"rho={rho0:g}",
            showlegend=False,
            hovertemplate=(
                f"<b>fixed rho={rho0:g}</b>"
                "<br>P=%{x:.5g}<br>Q=%{y:.5g}<br>R=%{z:.5g}"
                "<extra></extra>"
            ),
        )
    )

add_missing_curve(fig2)
show_figure(fig2, "Candidate 2: ruled surface by fixed-rho lines")

## Candidate 3 — fixed-\(R\) slice curves

Each colored curve lies in a horizontal plane \(R=c\). These are direct
coordinate-plane sections of the surface.

In [ ]:
fig3 = go.Figure()
add_surface_patch(fig3, opacity=0.08)

R_slices = [-1.2, -0.6, 0.0, 0.5, 1.0, 1.8, 2.8, 4.0]
rho = np.linspace(-3.6, 3.6, 2800)

for R0 in R_slices:
    c = np.full_like(rho, R0)
    P, Q, R = surface_coordinates(rho, c)
    P, Q, R = clip_xyz(P, Q, R)
    fig3.add_trace(
        go.Scatter3d(
            x=P, y=Q, z=R,
            mode="lines",
            line=dict(width=6),
            name=f"R={R0:g}",
            hovertemplate=(
                f"<b>R={R0:g}</b>"
                "<br>P=%{x:.5g}<br>Q=%{y:.5g}"
                "<extra></extra>"
            ),
        )
    )

add_missing_curve(fig3)
show_figure(fig3, "Candidate 3: fixed-R slice curves")

## Candidate 4 — fixed-\(P\) slice curves

For a fixed value \(P=P_0\), use
\[
Q=\rho+\frac{3P_0}{\rho},
\qquad
R=\frac{1}{\rho}-\frac{P_0}{\rho^3}.
\]
The positive and negative \(\rho\) branches are drawn separately.

In [ ]:
fig4 = go.Figure()
add_surface_patch(fig4, opacity=0.08)

P_slices = [-0.5, 0.0, 0.25, 0.7, 1.3, 2.2, 3.5, 4.5]
rho_abs = np.geomspace(0.04, 8.0, 2800)

for P0 in P_slices:
    first_branch = True
    for sign in (-1.0, +1.0):
        rho = sign * rho_abs
        P = np.full_like(rho, P0)
        Q = rho + 3.0*P0/rho
        R = 1.0/rho - P0/rho**3
        P, Q, R = clip_xyz(P, Q, R)

        fig4.add_trace(
            go.Scatter3d(
                x=P, y=Q, z=R,
                mode="lines",
                line=dict(width=5),
                name=f"P={P0:g}",
                legendgroup=f"P={P0:g}",
                showlegend=first_branch,
                hovertemplate=(
                    f"<b>P={P0:g}</b>"
                    "<br>Q=%{y:.5g}<br>R=%{z:.5g}"
                    "<extra></extra>"
                ),
            )
        )
        first_branch = False

add_missing_curve(fig4)
show_figure(fig4, "Candidate 4: fixed-P slice curves")

## Candidate 5 — fixed-\(Q\) slice curves

For a fixed value \(Q=Q_0\), use
\[
P=\frac{\rho(Q_0-\rho)}{3},
\qquad
R=\frac{4\rho-Q_0}{3\rho^2}.
\]

In [ ]:
fig5 = go.Figure()
add_surface_patch(fig5, opacity=0.08)

Q_slices = [-6.0, -4.0, -2.0, 0.0, 2.0, 4.0, 6.0]
rho_abs = np.geomspace(0.04, 8.0, 2800)

for Q0 in Q_slices:
    first_branch = True
    for sign in (-1.0, +1.0):
        rho = sign * rho_abs
        P = rho*(Q0-rho)/3.0
        Q = np.full_like(rho, Q0)
        R = (4.0*rho-Q0)/(3.0*rho**2)
        P, Q, R = clip_xyz(P, Q, R)

        fig5.add_trace(
            go.Scatter3d(
                x=P, y=Q, z=R,
                mode="lines",
                line=dict(width=5),
                name=f"Q={Q0:g}",
                legendgroup=f"Q={Q0:g}",
                showlegend=first_branch,
                hovertemplate=(
                    f"<b>Q={Q0:g}</b>"
                    "<br>P=%{x:.5g}<br>R=%{z:.5g}"
                    "<extra></extra>"
                ),
            )
        )
        first_branch = False

add_missing_curve(fig5)
show_figure(fig5, "Candidate 5: fixed-Q slice curves")

## Candidate 6 — two curve families as a coordinate net

This combines sparse fixed-\(\rho\) ruling lines with sparse fixed-\(R\)
slice curves. It may communicate “surface” more clearly than a solid patch.

In [ ]:
fig6 = go.Figure()

rho_lines = [-3.0, -2.25, -1.5, -0.8, -0.4,
              0.4, 0.8, 1.5, 2.25, 3.0]
c = np.linspace(R_RANGE[0], R_RANGE[1], 1000)

for i, rho0 in enumerate(rho_lines):
    P, Q, R = surface_coordinates(rho0, c)
    P, Q, R = clip_xyz(P, Q, R)
    fig6.add_trace(
        go.Scatter3d(
            x=P, y=Q, z=R,
            mode="lines",
            line=dict(width=5, color="#1f77b4"),
            name="fixed rho",
            legendgroup="fixed-rho",
            showlegend=(i == 0),
            hovertemplate=(
                f"<b>fixed rho={rho0:g}</b>"
                "<br>P=%{x:.5g}<br>Q=%{y:.5g}<br>R=%{z:.5g}"
                "<extra></extra>"
            ),
        )
    )

R_slices = [-1.0, 0.0, 0.75, 1.5, 2.5, 3.8, 4.8]
rho = np.linspace(-3.6, 3.6, 2800)

for i, R0 in enumerate(R_slices):
    P, Q, R = surface_coordinates(rho, np.full_like(rho, R0))
    P, Q, R = clip_xyz(P, Q, R)
    fig6.add_trace(
        go.Scatter3d(
            x=P, y=Q, z=R,
            mode="lines",
            line=dict(width=4, color="#d62728"),
            name="fixed R",
            legendgroup="fixed-R",
            showlegend=(i == 0),
            hovertemplate=(
                f"<b>fixed R={R0:g}</b>"
                "<br>P=%{x:.5g}<br>Q=%{y:.5g}"
                "<extra></extra>"
            ),
        )
    )

add_missing_curve(fig6, width=12)
show_figure(fig6, "Candidate 6: two-family coordinate net")

## Candidate 7 — ruling lines only

This removes the translucent surface entirely. It is the cleanest option for
a presentation slide or paper figure if the surface patch feels too cloudy.

In [ ]:
fig7 = go.Figure()

rho_values = np.linspace(-3.2, 3.2, 19)
rho_values = rho_values[np.abs(rho_values) > 0.15]
c = np.linspace(R_RANGE[0], R_RANGE[1], 1200)

for rho0 in rho_values:
    P, Q, R = surface_coordinates(rho0, c)
    P, Q, R = clip_xyz(P, Q, R)
    fig7.add_trace(
        go.Scatter3d(
            x=P, y=Q, z=R,
            mode="lines",
            line=dict(width=5),
            name=f"rho={rho0:.2f}",
            showlegend=False,
            hovertemplate=(
                f"<b>rho={rho0:.3g}</b>"
                "<br>P=%{x:.5g}<br>Q=%{y:.5g}<br>R=%{z:.5g}"
                "<extra></extra>"
            ),
        )
    )

add_missing_curve(fig7, width=13)
show_figure(fig7, "Candidate 7: ruling lines only")

## Selection notes

Record the candidate number and any camera angle that looks best.

Likely starting points:

- **Candidate 2** — best explanation of the ruled structure.
- **Candidate 3** — clearest conventional coordinate slices.
- **Candidate 6** — strongest “surface made from curves” picture.
- **Candidate 7** — cleanest publication-style line drawing.

After choosing one, the next notebook can place that exact family over the
full Artifact 24 range geometry.